# 🌡️ İklim Değişikliği Trend, Pre-Whitening ve SQMK Kırılma Analizi

**Yazar:** Mustafa Kerem Öztaş  
**Kurum:** Pamukkale Üniversitesi Fen Bilimleri Enstitüsü — İnşaat Mühendisliği A.D. (Hidrolik Bilim Dalı)

---

## 📌 Analiz Akışı ve Metodolojik Sıralama
Bu çalışmada uygulanan adımlar bilinçli bir hidrolojik mantık ve matematiksel sıralama ile yürütülmektedir:

1. **Veri Ön İşleme (Data Preprocessing):** Ham verideki eksiklikler temizlenir ve yıllık ekstrem değer dizisi çıkarılır.
2. **Sen's Slope (Sen'in Eğim Testi):** Zaman serisindeki yıllık ortalama değişim miktarı ($\beta$) hesaplanır.
3. **Trend-Free Pre-Whitening (TFPW):** Serideki otokorelasyon ($r_1$), Mann-Kendall testinde "yanlış pozitif" (Type-I hata) trend üretir. Bu nedenle önce trend seriden sökülür, otokorelasyon temizlenir ve eğim geri eklenir.
4. **Sequential Mann-Kendall (SQMK):** Otokorelasyondan arındırılmış veri üzerinde eğri kesişim yöntemiyle iklimsel kırılma yılları tespit edilir.

In [ ]:
# 1. GEREKLİ KÜTÜPHANELERİN YÜKLENMESİ
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Gerekli kütüphaneler başarıyla yüklendi.")

### Adım 1: Veri Yükleme ve Ön İşleme
* **Neden Yapılıyor?** Meteoroloji Genel Müdürlüğü (MGM) verileri karmaşık başlık yapısına sahiptir. Analizin doğru çalışabilmesi için verilerin sayısal matrise dönüştürülmesi ve her yılın maksimum değerinin çekilmesi gerekir.
* **Sıralama Mantığı:** Tüm istatistiksel analizlerin temeli temiz veri dizisidir.

In [ ]:
# Veri Seçimi ve Temizleme
print("Lütfen MGM Excel dosyanızı seçin:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

def load_and_clean_data(file_path, station_row_idx=1, data_rows=16):
    df_raw = pd.read_excel(file_path, skiprows=4)
    df_station = df_raw.iloc[station_row_idx:station_row_idx+data_rows, 1:14].copy()
    df_station.columns = ['Yıl', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
    
    df_station['Yıl'] = pd.to_numeric(df_station['Yıl'])
    for month in range(1, 13):
        df_station[str(month)] = pd.to_numeric(df_station[str(month)])
        
    years = df_station['Yıl'].values
    annual_max = df_station.iloc[:, 1:13].max(axis=1).values
    return years, annual_max

years, temp_series = load_and_clean_data(file_name)
print(f"Veri Yüklendi: {years[0]} - {years[-1]} yılları arası ({len(years)} veri noktası).")

### Adım 2: Sen'in Eğim Testi (Sen's Slope)
* **Neden Yapılıyor?** Serideki yıllık ortalama sıcaklık değişim oranını ($\beta$) parametrik olmayan (non-parametric) yoldan hesaplamak için.
* **Sıralama Mantığı:** TFPW (Pre-whitening) işleminde eğimi seriden geçici olarak sökmek gerektiğinden, ilk olarak $\beta$ değeri hesaplanmalıdır.
* **Matematiksel Formül:**  
  Tüm $i < j$ veri çiftleri için eğimler hesaplanır:  
  $$Q_k = \frac{x_j - x_i}{j - i}, \quad k = 1, 2, \dots, N$$  
  **Sen's Slope ($\beta$):** Hesaplanan $Q_k$ değerlerinin medyanıdır:  
  $$\beta = \text{median}(Q)$$

In [ ]:
def calc_sens_slope(x):
    n = len(x)
    slopes = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            slopes.append((x[j] - x[i]) / (j - i))
    return np.median(slopes)

beta = calc_sens_slope(temp_series)
print(f"Hesaplanan Sen's Slope Eğimi (β): {beta:.4f} °C/yıl")

### Adım 3: Trend-Free Pre-Whitening (TFPW)
* **Neden Yapılıyor?** Sıcaklık zaman serilerinde ardışık yıllar arasında 1. derece otokorelasyon ($r_1$) bulunabilir. Otokorelasyon, Mann-Kendall testinde var olmayan sanal bir trend varmış gibi görünmesine neden olur.
* **Sıralama Mantığı:** SQMK testine girmeden önce zaman serisi otokorelasyondan temizlenmelidir.
* **Matematiksel Formül ve Adımlar:**
  1. **Eğimi Sökme (Detrending):** $y_t = x_t - \beta \cdot t$
  2. **Lag-1 Otokorelasyon Hesabı:**  
     $$r_1 = \frac{\sum_{t=1}^{n-1} (y_t - \bar{y})(y_{t+1} - \bar{y})}{\sum_{t=1}^{n} (y_t - \bar{y})^2}$$
  3. **AR(1) Arındırma ve Eğimi Geri Ekleme:**  
     $$y'_t = y_t - r_1 \cdot y_{t-1}$$
     $$x^{TFPW}_t = y'_t + \beta \cdot t$$

In [ ]:
def tfpw_prewhitening(x, beta):
    n = len(x)
    t = np.arange(1, n + 1)
    
    # 1. Eğimi sök
    y = x - beta * t
    
    # 2. Lag-1 Otokorelasyon
    y_mean = np.mean(y)
    r1 = np.sum((y[:-1] - y_mean) * (y[1:] - y_mean)) / np.sum((y - y_mean) ** 2)
    
    # 3. Temizle ve Eğimi Geri Ekle
    y_prime = y[1:] - r1 * y[:-1]
    x_tfpw = y_prime + beta * t[1:]
    return x_tfpw, r1

tfpw_series, r1 = tfpw_prewhitening(temp_series, beta)
print(f"Lag-1 Otokorelasyon Kat Sayısı (r1): {r1:.4f}")

### Adım 4: Sequential Mann-Kendall (SQMK) ve Görselleştirme
* **Neden Yapılıyor?** İklimsel kırılma/rejim değişim yılını belirlemek için. İleri ($UF$) ve geri ($UB$) istatistik eğrilerinin kesiştiği nokta kırılma yılını gösterir.
* **Sıralama Mantığı:** Otokorelasyondan arındırılmış `tfpw_series` veri seti üzerine uygulanır.
* **Matematiksel Formül:**  
  $i$. elemandan önceki kendisinden küçük elemanların sayısı $n_i$ olmak üzere toplam test istatistiği $t_k$:  
  $$t_k = \sum_{i=1}^{k} n_i$$  
  Beklenen değer $E(t_k)$ ve Varyans $Var(t_k)$:  
  $$E(t_k) = \frac{k(k-1)}{4}, \quad Var(t_k) = \frac{k(k-1)(2k+5)}{72}$$  
  Standartlaştırılmış İleri Test İstatistiği $u(t)$:  
  $$u(t) = \frac{t_k - E(t_k)}{\sqrt{Var(t_k)}}$$

In [ ]:
def sequential_mann_kendall(x):
    n = len(x)
    def calculate_u(series):
        u = np.zeros(n)
        t_stat = 0
        for k in range(1, n):
            n_k = np.sum(series[:k] < series[k])
            t_stat += n_k
            E_t = (k + 1) * k / 4.0
            Var_t = (k + 1) * k * (2 * (k + 1) + 5) / 72.0
            u[k] = (t_stat - E_t) / np.sqrt(Var_t)
        return u

    uf = calculate_u(x)
    ub = calculate_u(x[::-1])[::-1] * -1.0
    return uf, ub

uf, ub = sequential_mann_kendall(tfpw_series)

# TFPW nedeniyle 1 yıl azalan zaman dizisini hizalama
years_tfpw = years[1:]

# GRAFİK ÇİZİMİ
plt.figure(figsize=(10, 5), dpi=150)
plt.plot(years_tfpw, uf, 'r-o', label='UF (İleri İstatistik)', linewidth=2)
plt.plot(years_tfpw, ub, 'b--s', label='UB (Geri İstatistik)', linewidth=2)
plt.axhline(y=1.96, color='k', linestyle='--', alpha=0.7, label='+1.96 (%95 Güven Sınırı)')
plt.axhline(y=-1.96, color='k', linestyle='--', alpha=0.7, label='-1.96 (%95 Güven Sınırı)')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

plt.title("SQMK Kırılma ve Trend Analizi Grafik Çıktısı", fontsize=12)
plt.xlabel("Yıllar", fontsize=10)
plt.ylabel("Z İstatistiği u(t)", fontsize=10)
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()